Model Comparison Script 

In [ ]:
# import necessary libraries 
import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split 

import pickle

In [ ]:
# preparing training/testing data 
features_all = pd.read_pickle("training_features_11032026.pkl")

X_zygo = features_all[["Zygo"]] 
X_corr = features_all[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all["Num_Contractions_Corr"].astype(int).to_numpy()])       

   
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # keep 20% purely for testing 

In [ ]:
results = [] 

models_folder = "models"   # change to your folder path

model_files = sorted(glob.glob(os.path.join(models_folder, "*.pkl")))

for model_file in model_files:
    model_name = os.path.splitext(os.path.basename(model_file))[0]
    
    # skipping unwanted models 
    if (model_name.startswith("M") and model_name[1:].isdigit() and "1" not in model_name) == False:
        continue 

    with open(model_file, "rb") as f:
        model = pickle.load(f)

   
    # predict 
    y_pred_train = model.predict(X_train)  
    y_pred_train = np.argmax(y_pred_train, axis=1)   

    # Predict on test data
    y_pred_test = model.predict(X_test)   
    y_pred_test = np.argmax(y_pred_test, axis=1)   

    # Calculate accuracy
    accuracy_training = accuracy_score(y_train, y_pred_train)   
    accuracy_test = accuracy_score(y_test, y_pred_test)  

    # Calculate F1 score
    f1_training = f1_score(y_train, y_pred_train, average='weighted')  
    f1_test = f1_score(y_test, y_pred_test, average='weighted')  

    # Calculate precision 
    prec_training = precision_score(y_train, y_pred_train, average="weighted", zero_division=0)
    prec_test = precision_score(y_test, y_pred_test, average="weighted", zero_division=0)

    # Calculate recall 
    recall_training = recall_score(y_train, y_pred_train, average="weighted", zero_division=0)
    recall_test = recall_score(y_test, y_pred_test, average="weighted", zero_division=0)



    results.append({
        "model": model_name,
        "accuracy_train": accuracy_training,
        "accuracy_test": accuracy_test,
        "precision_train": prec_training,
        "precision_test": prec_test,
        "recall_train": recall_training,
        "recall_test": recall_test,
        "f1_train": f1_training, 
        "f1_test": f1_test
    })


#metrics_df = pd.DataFrame(results)

#metrics_df.to_excel("model_comparison_metrics.xlsx", index=False)
#print(metrics_df)

In [ ]:
metrics = ["accuracy_train", "precision_train", "recall_train", "f1_train"]

for metric in metrics:
    plt.figure(figsize=(10, 5))
    plt.bar(metrics_df["model"], metrics_df[metric])
    plt.xlabel("Model")
    plt.ylabel(metric.capitalize())
    plt.title(f"{metric.capitalize()} Comparison Across Models")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()